In [20]:
import pandas as pd
import numpy as np
import io

import seaborn as sns
import matplotlib.pyplot as plt
from pymatgen.core.structure import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pyxtal import pyxtal
from tqdm import tqdm
import scienceplots
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from contextlib import redirect_stdout

plt.style.use(["science", "ieee", "no-latex"])

## Plot heatmap of wyckoff and atom distribution

In [8]:
test_data = pd.read_csv("/home/wangqc/project/LEGO-xtal/data/train/mp_20/process/aug_V3/train_cif.csv")
w_keys = ['4a', '4b', '8c', '24d', '24e', '32f', '48g', '48h', '48i']
w_values = [i for i in range(1,10)]
w_dict = {key: value for key,value in zip(w_keys, w_values)}

e_keys= [
    # 1
    'H', 'He',
    # 2
    'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne',
    # 3
    'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar',
    # 4
    'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr',
    # 5
    'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
    'In', 'Sn', 'Sb', 'Te', 'I', 'Xe',
    # 6
    'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy',
    'Ho', 'Er', 'Tm', 'Yb', 'Lu',
    'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi',
    'Po', 'At', 'Rn',
    # 7
    'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu']
e_values = [i for i in range(1,95)]
e_dict = {key: value for key,value in zip(e_keys, e_values)}

In [21]:
trap = io.StringIO()
def process_one(cif_string):

    with redirect_stdout(trap):
        st = Structure.from_str(cif_string, fmt='cif')
    spga = SpacegroupAnalyzer(st)
    crystal = spga.get_refined_structure()
    c = pyxtal()
    try:
        c.from_seed(crystal, tol=0.01)
    except:
        c.from_seed(crystal, tol=0.0001)

    e_w = []
    for site in c.atom_sites:
        wyckoff = str(site.wp.multiplicity) + str(site.wp.letter)
        ele = site.specie
        if w_dict.get(wyckoff, False) and e_dict.get(ele, False):
            wyckoff = w_dict.get(wyckoff)
            ele = e_dict.get(ele)
            e_w.append((ele, wyckoff))
    return e_w


def plot_heat_map(data):
    '''
    data.shape is (9, 94)
    '''
    plt.figure(figsize=(14, 2))
    norm = LogNorm(vmin=0.1, vmax=data.max())
    cmap = sns.color_palette("hot", as_cmap=True)  
    sns.heatmap(data, annot=False, cmap=cmap.reversed(), cbar=True, norm=norm)
    plt.title("Heatmap")
    plt.show()
    

In [22]:
from joblib import Parallel, delayed
#test_data = test_data[test_data["spacegroup.number"] == 225]
e_w = Parallel(100, backend="multiprocessing")(delayed(process_one)(test_data["cif"].iloc[i]) for i in tqdm(range(len(test_data))))

data = np.zeros((9,94))
for i in e_w:
    if i != []:
        for (e,w) in i:
            data[w-1][e-1] += 1

plot_heat_map(data)

  0%|          | 199/367567 [01:41<52:12:22,  1.95it/s]
/home/wangqc/miniconda3/envs/lego/lib/python3.11/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/wangqc/miniconda3/envs/lego/lib/python3.11/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/wangqc/miniconda3/envs/lego/lib/python3.11/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/wangqc/miniconda3/envs/lego/lib/python3.11/site-packages

KeyboardInterrupt: 

## Plot Fig F2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 创建一些示例数据
x = np.arange(1, 221)
y1 = np.random.rand(220)  # 结构有效性数据
y2 = np.random.rand(220)  # 组成有效性数据
y3 = np.zeros(220)
for i in [10, 30, 50, 70, 90, 110, 130, 150, 170, 190, 210]:
    y3[i-10:i] = np.random.randint(1, 15, size=10)  # 空间群频率数据

def plot_f2(g, struc_valid, comp_valid, g_number):
    fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    axs[0].bar(g, struc_valid, color='blue', alpha=0.7)
    axs[0].set_ylabel('Structure Validity')

    axs[1].bar(g, comp_valid, color='red', alpha=0.7)
    axs[1].set_ylabel('Composition Validity')

    axs[2].bar(g, g_number, color='green', alpha=0.7)
    axs[2].set_xlabel('Spacegroup Number')
    axs[2].set_ylabel('Frequency (%)')

    for ax in axs:
        ax.set_title('')

    plt.subplots_adjust(hspace=0)
    plt.show()

TypeError: plot_f2() missing 4 required positional arguments: 'g', 'struc_valid', 'comp_valid', and 'g_number'